# Experiment 11 — risk estimation, weighting and position sizing

Based on [experiment 10](10-research-portfolio-snapshots.ipynb) and the [reviewed plan](11-weighting-sizing-plan.md), co-designed with Claude CLI fable 5.1.

## Goal and method

Improve weighting and sizing while keeping the selected vaults fixed. Determine whether inverse-variance performance comes from its risk horizon, strength of the allocation preference, shrinkage towards a peer prior, score transformation or subsequent entry constraints. Seek a sensible parameter neighbourhood rather than one best historical cell. Freeze six-vault **focal_four** and **v11** baskets, their eligibility, missing-score tiers and tie-breaking.

Compare risk windows 45/90/180 days, allocation powers 1/1.5/2, fixed kappa 30 versus inherited L/3, raw and regularised risk, bounded inverse-risk scores, median/maximum multi-window forecasts and matched calendar-log/gap-scaled sparse estimators. Risk is calculated on the full eligible universe before selecting baskets. Only NAV is causally forward-filled; actual updates and permission/capacity clocks remain separate. Young/partial histories use the declared peer prior and whole-basket equal fallback, without changing selection.

## Calendar and scope

**2026-01-01 → 2026-10-05, exclusive end, naive UTC**. Four-day cadence gives **70 origins**. Features end at T−1; all 2025 and earlier warmup is retained. Fixed units use inherited anchor NAV(T−1) and terminal NAV(T−1+h). This convention ensures NB10 parity; it is not an executable claim of filling yesterday’s NAV at today’s decision. Mature counts are **68/62/47** for 7/30/90 days, with 2/8/23 pending. Targets require vault-specific terminal support; unresolved holdings stay unresolved.

The **18-policy smoke** is a correctness checkpoint over the full calendar at 30 days. After it passes, the **40-policy main study** reuses those allocations and evaluates all three horizons: 2,800 allocations and 8,400 scheduled outcomes. Twelve sensitivities and eight cap-aware policies are registered, not executed. Historical outcomes are descriptive: primary 30-day inference has at most eight complete blocks, below the required ten. Overlapping snapshots must not be compounded into strategy CAGR.

## Key new insights

The original focal 90/p2 policy leads all nine core window/power cells. Gap-scaled focal sizing gives **2.579% mean 30-day return versus 2.512% for the baseline**, but its median is lower. T18 improves on V180-only sizing by **0.359 percentage points**, chiefly in Q1. Risk calibration and profitable sizing do not always agree. Cash is identical across sizing methods within each selector.

## Summary of results

Completed **18-policy smoke and 40-policy main**, all 70 origins and 7/30/90-day horizons: **2,800 allocations / 8,400 scheduled outcomes**. Mature/resolved counts are 68/68, 62/62 and 47/46. Exact historical controls reproduce. All **17 numerical fixtures,22 smoke and26 main checks pass**. The notebook has 27 separate main charts plus one smoke chart and no logged warnings. Twelve sensitivities and eight cap-aware policies remain registered, unexecuted.

## Robustness of results

All 2025/earlier warmup is retained. No future availability controls selection. Sparse/partial/flat fixtures, exact source controls, future-state mutation, signed unresolved bounds, capacity conservation and same-budget effects pass. The 17 January 90-day Dobutsu–Alpha Club target remains unresolved rather than inventing future NAV. Primary inference has only eight complete blocks, so conclusions remain descriptive. Vault effects offset strongly; T18's gain is regime-sensitive. Fixed-unit anchor fills and retrospective inventory/publication clocks limit production interpretation.

See [findings](11-weighting-sizing-findings.md) and [folder summary](summary.md#nb11--risk-estimation-weighting-and-position-sizing) for hypotheses, outcomes and next steps.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown
ROOT=Path.cwd()
if not (ROOT/'sizing_study_design.py').exists():
    ROOT=ROOT/'scratchpad/hyperliquid-vov-2nd'
sys.path.insert(0,str(ROOT))
from sizing_study_design import smoke_registry, main_registry, followups
from sizing_study_data import read_inputs, stage_folder, freeze_baskets, build_risk
from sizing_study_analysis import execute
from sizing_study_audit import fixtures, audit
from sizing_study_report import render_smoke, render_main, write_findings
from score_matrix_visualisation import compact_output
pd.options.display.max_colwidth=None


## Independent mathematical fixtures — 2026-01-01 → 2026-10-05; 30-day horizon; smoke

Check sparse/partial/flat risk, ddof1, priors, shrinkage, log-space powers, gap drift, fixed-unit paths, unresolved bounds and exact registry counts independently before inspecting returns.


In [2]:
fixture_results=fixtures()
display(fixture_results)


,check,passed,description
0,Partial and sparse simple variance,True,"Three elapsed returns [0.1,0,0.1], two actual updates, ddof1"
1,Flat versus unavailable,True,An actual update is needed; filled zeros alone are insufficient
2,Positive leave-one-out full prior,True,"Self, flat and missing excluded; complete pool preferred"
3,Fixed shrinkage and prior-only fallback,True,Thirty real updates gives half empirical weight; missing gives none
4,Inherited kappa companion,True,L180 inherited strength differs from core kappa30
5,Raw floor asymmetry,True,Regularised estimator has no raw variance floor
6,Log-space allocation power,True,Inverse volatility and variance odds calculated independently
7,Whole-basket equal fallback,True,Unavailable risk retains every selected name
8,Zero-request preserving cap projection,True,Cap redistributes only to positive requests and conserves budget
9,Fixed-unit arithmetic,True,"Cash0.1 plus initial units, no daily rebalancing"


## Pinned inputs and full warmup — 2026-01-01 → 2026-10-05; 30-day horizon; smoke

Verify source archives, metadata, 139 recipes, NB09 features and NB10 component/code hashes. Freeze both baskets once; do not use future availability to alter selection. Earlier generations are read-only.


In [3]:
with compact_output('Read-only inputs and causal risk; estimated one–two minutes'):
    data=read_inputs()
    baskets=freeze_baskets(data)
    risk,support=build_risk(data)
display(pd.DataFrame([{'Origins':len(data['origins']),'Recorded NAV start':str(data['daily'].date.min()),'Frozen recipes':len(data['registry']),'Unique frozen baskets':baskets.groupby(['origin_at','selector']).ngroups,'Unique risk estimates':len(risk)}]))


Read-only inputs and causal risk; estimated one–two minutes — complete

,Origins,Recorded NAV start,Frozen recipes,Unique frozen baskets,Unique risk estimates
0,70,2023-03-01 00:00:00,139,140,114318


## Full-calendar smoke checkpoint — 2026-01-01 → 2026-10-05; 30-day horizon; smoke

Run all 18 registered methods at 70 dates, then compare exact saved controls and per-vault ledgers. Advance based on correctness rather than profitability. A separate chart shows the earliest mechanically cap-active example.


In [4]:
smoke_policies=smoke_registry()
stage_folder(data,smoke_policies,'smoke')
with compact_output('18-policy full-calendar smoke; estimated under two minutes'):
    smoke=execute(data,baskets,risk,smoke_policies,[30])
    smoke_audit=audit(data,smoke,smoke_policies,[30])
smoke_figures=render_smoke(data,smoke,smoke_audit)
display(pd.DataFrame([{'Smoke checks passed':int(smoke_audit.passed.sum()),'Numerical fixtures passed':int(fixture_results.passed.sum()),'Smoke allocations':18*70,'Smoke target rows':len(smoke['outcomes'])}]))


18-policy full-calendar smoke; estimated under two minutes — complete

### Saved control reproduction — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon; smoke

Exact NB10 per-date weights and outcomes, using unchanged archive and feature fingerprints.

arm,reference,reference_origins,new_origins,outcomes,weights_match,outcomes_match,mean_return
S01,T01,70,70,62,True,True,1.72%
S14,T00,70,70,62,True,True,1.47%
S11,T03,70,70,62,True,True,2.38%
S16,T18,70,70,62,True,True,2.39%
S03,Original NB10 focal regularised90/p2,70,70,62,True,True,2.51%


### 18-policy smoke checkpoint — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon; smoke

Full calendar is retained; 62 mature 30-day origins per arm. This checks implementation, not profitability.

arm,label,resolved,mean_return,mean_variance,mean_cash,effective_N
S01,F:equal,62,1.72%,0.0001956,3.32%,5.188
S02,L:equal,62,1.93%,0.0001688,3.01%,5.242
S03,F:regular L90/p2/κ30,62,2.51%,0.0001195,3.32%,4.311
S04,L:regular L90/p2/κ30,62,1.99%,9.977e-05,3.01%,4.384
S05,F:regular L45/p1/κ30,62,1.96%,0.0001313,3.32%,4.781
S06,L:regular L45/p1/κ30,62,1.76%,0.0001145,3.01%,4.85
S07,F:regular L180/p1.5/κ30,62,2.03%,0.0001459,3.32%,4.652
S08,L:regular L180/p1.5/κ30,62,1.97%,0.0001214,3.01%,4.702
S09,F:raw L90/p2/κ30,62,2.35%,0.0001179,3.32%,4.025
S10,L:raw L90/p2/κ30,62,1.47%,0.0001004,3.01%,4.052


### Smoke cap-active example — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon; smoke

Earliest focal baseline date with positive cap distortion: 2026-01-01. Chosen from allocation mechanics before inspecting returns.

,Smoke checks passed,Numerical fixtures passed,Smoke allocations,Smoke target rows
0,22,17,1260,1260


## Complete main matrix — 2026-01-01 → 2026-10-05; 7/30/90-day horizons; main

Reuse all 18 smoke allocations and add the 22 remaining registered methods. Allocate once per origin/policy; only future valuation changes with horizon. Save pending dates, terminal censoring, calibration and matched effects.


In [5]:
policies=main_registry()
stage_folder(data,policies,'main')
with compact_output('40 policies,70origins,three horizons; estimated two–five minutes'):
    result=execute(data,baskets,risk,policies,[7,30,90],reuse=smoke)
    main_audit=audit(data,result,policies,[7,30,90],reuse=smoke)
    support.to_parquet(data['artifact']/'risk_pool_support.parquet',index=False)
display(pd.DataFrame([{'Policies':len(policies),'Origin-policy allocations':40*70,'Scheduled outcomes':len(result['outcomes']),'Main audit checks passed':int(main_audit.passed.sum()),'Execution seconds':data['execution_seconds']}]))


40 policies,70origins,three horizons; estimated two–five minutes — complete

,Policies,Origin-policy allocations,Scheduled outcomes,Main audit checks passed,Execution seconds
0,40,2800,8400,26,26.455589


## Results and mechanisms — 2026-01-01 → 2026-10-05; 30-day primary horizon; main

Every table and figure has a separate heading and explanation. Main displays use 30 days; explicitly titled 7/90-day tables are secondary. Colour scales match across comparable cells, and pending/unsupported cells remain grey.


In [6]:
figures=render_main(data,result,main_audit,policies,support)


### Calendar and maturity — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 7/30/90-day horizon; main

All 70 origins retained at each horizon; features use T−1 and full 2025/earlier warmup. Pending targets are never zero returns.

horizon,scheduled,mature,pending
7,70,68,2
30,70,62,8
90,70,47,23


### Maturity versus terminal support — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 7/30/90-day horizon; main

Administrative maturity does not guarantee supported terminal NAV. Seven/30-day outcomes resolve 68/62 dates; 90-day resolves 46 of 47 mature origins. Unsupported invested weight remains unresolved.

arm,selector,h,scheduled,mature,pending,resolved
S03,F,7,70,68,2,68
S03,F,30,70,62,8,62
S03,F,90,70,47,23,46
S04,L,7,70,68,2,68
S04,L,30,70,62,8,62
S04,L,90,70,47,23,46


### Unresolved 90-day target — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 90-day horizon; main

Selected using only past information. The target ends beyond this vault’s final actual positive mark; NAV is not carried past terminal support for target evaluation.

...

arm,control,quarter,paired,mean_difference,first_update_difference
M40,S04,2026Q1,23,0.00%,0.03%
M40,S04,2026Q2,23,0.04%,-0.01%
M40,S04,2026Q3,16,0.00%,0.01%
S16,S15,2026Q1,23,1.11%,0.08%
S16,S15,2026Q2,23,-0.14%,0.03%
S16,S15,2026Q3,16,-0.00%,0.03%
S18,S03,2026Q1,23,0.02%,0.03%
S18,S03,2026Q2,23,0.05%,-0.01%
S18,S03,2026Q3,16,0.16%,0.06%


### Selector dependence — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon; main

All scheduled origins. Basket identity overlap and shared capital are continuous descriptive measures; selectors are related robustness contexts.

mean_Jaccard,min_Jaccard,mean_shared_accepted_weight
88.27%,50.00%,84.61%


### Next-stage inventory — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon; main

Exact policies saved in followups.json. No return-based automatic expansion and no shuffled-weight placebos.

batch,policies,status
Local/prior/floor sensitivities,12,"registered, not run"
Cap-aware convex allocation,8,"registered, not run"


## Verdict and registered next work — 2026-01-01 → 2026-10-05; 30-day horizon; main

Write findings only after the 40-policy study and its exact control checks finish. No sensitivities or cap-aware follow-up optimisation are run. Results describe searched historical snapshots, not an executable production return.


In [7]:
findings_path=write_findings(data,result,main_audit)
display(Markdown(f'[Detailed findings]({findings_path.name})'))
display(pd.DataFrame([{'Main checks passed':int(main_audit.passed.sum()),'Separate main figures':len(figures),'Sensitivity policies not run':len(followups()['sensitivities']),'Cap-aware policies not run':len(followups()['cap_aware']),'Generation':data['fingerprint']}]))


[Detailed findings](11-weighting-sizing-findings.md)

,Main checks passed,Separate main figures,Sensitivity policies not run,Cap-aware policies not run,Generation
0,26,27,12,8,b64b9738eff5af379f16d62b65e0147ce2ee4ab8143505b6fb42c864e37f161c
